# Lag effect Analysis

### "정책 효과가 강하게 나타난 시점일수록, 모델은 더 쉽게 결과(Y)를 예측할 수 있다"
1. 정책이 효과를 발휘하면, Y(조직성과/개인결과)가 X(정책포함 정보)에 더 강하게 종속된다.
- 정책 도입 전에는 조직성과(Y)가 다양한 외적 요인에 따라 들쭉날쭉 (예측 어려움).
- 정책이 효과를 발휘하면 → 특정 정책 조건(X)에 따라 Y가 더 일관되고 예측 가능하게 변화함.
- 즉, 정책이 Y에 설명력을 부여한다.

2. 모델은 설명력 높은 관계일수록 예측 성능이 높다.
- 머신러닝 모델의 본질: 입력 X와 출력 Y 사이의 규칙성을 찾아내는 것.
- 따라서, 만약 정책 효과로 인해 X (정책포함) → Y의 인과 경로가 강해졌다면,
- 모델은 더 쉽게 일반화 가능한 패턴을 학습함.
- 결과적으로, 정확도, F1, AUC 등의 성능이 높아짐.

1. Stage 1
    - 2020년 데이터 X + 2021년 데이터 y => 모델-2020_2021
    - 2020년 데이터 X + 2023년 데이터 y => 모델-2020_2023
    => 각 모델의 퍼포먼스 측정 (강한 예측력을 보이는 모델 = 그게 결국 영향력)
    => 각 모델 해석력 지수 얻기 / 중요도에 따라서 인풋값의 가중치 다르게 줌


2. Stage 2
    - 2021년 데이터 X + 2021년 데이터 y => 모델-2021_2021
    - 2021년 데이터 X + 2023년 데이터 y => 모델-2022_2023

3. Stage 3
    - 2022년 데이터 X + 2023년 데이터 y => 모델-2022_2023

4. Stage 4
    - 2023년 데이터 X + 2023년 데이터 y => 모델-2023_2023

---

- Dataset: 2020 / 2021 / 2022 / 2023, 근데 2021 이랑 2023 데이터만 target label 갖고 있음
- Y : Target label
    - 2021 : C21C05_01H1 (8) 전체 숙련수준  / C21C05_01H2 (8) 전체 경쟁력
    - 2023 : C23C05_01H1 (8) 전체 숙련수준  / C23C05_01H2 (8) 전체 경쟁력

- X :

In [6]:
import matplotlib.pyplot as plt
import utils_, config
#from github.V1.utils_ import data_processing
import os
import numpy as np
import pandas as pd
from collections import defaultdict



def get_all_data(year_list):
    file_names = config.file_names       # Expected file names (e.g., ['file1.csv', 'file2.csv'])
    file_list = os.listdir(config.path)  # All files in the directory
    dataset = {}                         # Final dictionary to store data
    year_cnt = -1                        # Counter to map years to files

    for expected_file in file_names:
        for actual_file in file_list:
            if expected_file == actual_file:
                year_cnt += 1
                current_year = year_list[year_cnt]
                print(f"{expected_file} ===> {current_year} data")

                # Load data
                df, meta = utils_.data_import(os.path.join(config.path, expected_file))

                # Store in dataset dict with year as key
                dataset[current_year] = {
                    'data': df,
                    'meta': meta
                }

    return dataset

In [8]:
year_list = config.year               # List of years (e.g., [2018, 2020, 2022])

dataset = get_all_data(year_list)

HCCP_2ndWave_Head_1st(최종).sav ===> 2020 data
HCCP_2ndWave_Head_2nd(최종).sav ===> 2021 data
HCCP_2ndWave_Head_3rd(최종).sav ===> 2022 data
HCCP_2ndWave_Head_4th.sav ===> 2023 data


In [9]:
for year in year_list:
    print(dataset[year]['data'].shape)

(510, 460)
(500, 389)
(500, 474)
(500, 392)


In [10]:
utils_.see_col_idx_and_name(dataset[year]['data'], dataset[year]['meta'])

C23_ID1 None
C23_IND1 산업분류
C23_SCALE 기업규모
C23_KSIC1 표준산업분류_대분류
C23_KSIC2 표준산업분류_중분류
C23_KSIC3 표준산업분류_소분류
C23_KSIC4 표준산업분류_세분류
C23_TYPE 기업형태
C23_SEX1 대표자성별 1
C23_SEX2 대표자성별 2
C23_SEX3 대표자성별 3
PANEL None
C23A01_01 모회사/자회사 여부
C23A01_02 외국인 지분 여부
C23A01_021 외국인 지분 역할
C23A01_03 2020년 매출 대비 수출 비중
C23A01_04 경영체제(오너경영/전문경영) 형태
C23A01_05 연구개발 조직 형태
C23A01_06 지난해 신제품 및 개선제품의 출시 여부
C23A01_061 신제품의 개발자
C23A01_062A 신제품의 시장출시로 인한 효과-1순위
C23A01_062B 신제품의 시장출시로 인한 효과-2순위
C23A01_07A 신제품 또는 개선제품 출시를 방해한 요인-1순위
C23A01_07B 신제품 또는 개선제품 출시를 방해한 요인-2순위
C23A02_01 2년전과 비교한 지난해 주력제품(상품/서비스) 수요 변화
C23A02_02 2년전과 비교한 신제품(상품/서비스)의 개발 및 도입
C23A02_021 지난해 기준 동종업계 대비 신제품 개발 및 도입 수준
C23A02_03 2년전과 비교한 지난해 기술변화
C23A02_031 지난해 기준 동종업계 대비 기술 변화 수준
C23A02_04 2년전과 비교한 지난해 본사 내의 부서 및 조직 변화
C23A02_05 2년전과 비교한 귀사 사업장의 라인이나 설비 변화나 지점/영업소 증감
C23A02_06 주력제품(상품/서비스) 시장 전략
C23A02_07 최우선 경영 방침 순위
C23A02_08A 해외 법인 기능_생산
C23A02_08B 해외 법인 기능_판매
C23A02_08C 해외 법인 기능_개발
C23A02_08D 해외 법인 기능_물류
C23A02_08E 해외 법인 기능_기타
C23A02_08F 해외 법인 기능_해외

In [19]:

#df['C23C05_01H1'].value_counts()


C23C05_01H1
3.0    387
4.0     92
2.0     17
1.0      2
5.0      2
Name: count, dtype: int64

In [1]:
import utils_, config, model

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

def main():
    for i in config.Target_Indices:

        aadf, meta = utils_.data_import(config.file_path)
        print(f"\n==============================================\n CLASS: {meta.column_labels[i]}\n==============================================\n\n")
        X, Y = utils_.data_processing(df, target_idx=i)
        #X = HRD_col_select(all_X)
        X_train, X_test, y_train, y_test= utils_.data_aug_smote(X, Y)
        col_name = df.columns[i]

        GB_model = model.GB(X_train, X_test, y_train, y_test)
        XGB_model = model.XGBoost(X_train, X_test, y_train, y_test, col_name)
        #LGBM_model = models_compare_all.LightGBM(X_train, X_test, y_train, y_test)
        #CatBoost_model = models_compare_all.CatBoost(X_train, X_test, y_train, y_test)
        #RF_model = models_compare_all.RF(X_train, X_test, y_train, y_test)
        #DT = models_compare_all.Dec_T(X_train, X_test, y_train, y_test)

        model.feature_importance(XGB_model, X, df, meta)


    return None

In [ ]:
if __name__ == "__main__":
    main()